# RFB bronze — exploração das tabelas Delta

**Como abrir:** VS Code no odyssey → *Remote-SSH* em `lambdaserver` → abra a pasta `~/lambda4` → este notebook →
*Select Kernel* → **Python (lambda-data)**.

Três jeitos de ler as tabelas Delta, cada um com seu papel:

| Ferramenta | Use para | Evite para |
|---|---|---|
| **DuckDB** (`duckdb_lake()`) | explorar, contar, agrupar, amostrar: rápido, sem JVM | escrever nas tabelas |
| **PySpark** | o mesmo motor do pipeline: histórico, time travel, CDF, `MERGE` | uma contagem rápida (sobe uma JVM) |
| **DBeaver** | `audit` no Postgres (execuções, checks, logs) | ler Delta (não fala com o log) |

Regra de ouro: **sempre filtre por `_ref_month`** (é a partição) e use `LIMIT` ao ver linhas: são 73 milhões.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))           # importa pipelines.commons a partir do repositório

REF = 202609                            # mês de referência (YYYYMM)

## 1. DuckDB: as tabelas com os mesmos nomes do Spark

`duckdb_lake()` lê o metastore Hive e cria uma *view* por tabela Delta (`rfb_bronze.socios`, `ctrl.download_ref`...),
apontando para o SeaweedFS via `delta_scan`.

In [ ]:
from pipelines.commons.lake import duckdb_lake, list_views

con = duckdb_lake()
list_views(con)

In [ ]:
# linhas por tabela no mês: count(*) sem filtro extra vem das estatísticas do log do Delta (instantâneo)
tables = [v for v in list_views(con) if v.startswith("rfb_bronze.")]
sql = " UNION ALL ".join(f"SELECT '{t}' AS tabela, count(*) AS linhas FROM {t} WHERE _ref_month = {REF}" for t in tables)
con.sql(sql + " ORDER BY linhas DESC").df()

In [ ]:
# amostra: escolha colunas (a bronze é toda string, exatamente como a Receita publicou)
con.sql(f"""
    SELECT cnpj_basico, cnpj_ordem, cnpj_dv, situacao_cadastral, data_inicio_atividade, cnae_fiscal_principal,
           municipio, uf, _source_file, _ingested_at
    FROM rfb_bronze.estabelecimentos
    WHERE _ref_month = {REF}
    LIMIT 5
""").df()

In [ ]:
# perfil rápido: situação cadastral (01 nula, 02 ativa, 03 suspensa, 04 inapta, 08 baixada)
con.sql(f"""
    SELECT situacao_cadastral, count(*) AS estabelecimentos,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM rfb_bronze.estabelecimentos WHERE _ref_month = {REF}
    GROUP BY 1 ORDER BY 2 DESC
""").df()

In [ ]:
# os casos "difíceis" do CSV chegaram intactos: quebra de linha dentro de aspas e aspas dobradas
con.sql(f"""
    SELECT cnpj_basico, cnpj_ordem, nome_fantasia, complemento
    FROM rfb_bronze.estabelecimentos
    WHERE _ref_month = {REF} AND (contains(nome_fantasia, chr(10)) OR contains(complemento, '"'))
    LIMIT 8
""").df()

In [ ]:
# controle: o que foi baixado e o que cada camada aplicou
display(con.sql("SELECT pipeline, ref_month, status, files_ok, round(bytes_total/2^30, 2) AS gib, attempts, downloaded_at FROM ctrl.download_ref").df())
con.sql("SELECT pipeline, layer, ref_month, status, rows, applied_at FROM ctrl.layer_watermark ORDER BY layer").df()

## 2. PySpark: o mesmo motor do pipeline

Use quando precisar do que só o Delta via Spark faz: **histórico de versões**, **time travel**, **Change Data Feed**.
`local[4]` roda o Spark dentro do notebook com 4 núcleos, sem ocupar o cluster (que fica livre para o Airflow).

In [ ]:
from pipelines.commons.spark import build_spark

spark = build_spark("notebook-explore", master="local[4]")
spark.sql("SHOW TABLES IN rfb_bronze").show()

In [ ]:
# histórico: cada escrita é uma versão, com métricas do commit
spark.sql("DESCRIBE HISTORY rfb_bronze.socios").select(
    "version", "timestamp", "operation", "operationParameters.mode", "operationMetrics.numOutputRows"
).show(truncate=False)

In [ ]:
# Change Data Feed: toda mudança de status do controle de downloads (DOWNLOADING -> DOWNLOADED...)
spark.sql("SELECT ref_month, status, attempts, _change_type, _commit_version FROM table_changes('ctrl.download_ref', 0) ORDER BY _commit_version").show()

In [ ]:
# time travel: como a tabela de controle estava na versão 1
spark.sql("SELECT ref_month, status, attempts FROM ctrl.download_ref VERSION AS OF 1").show()

In [ ]:
spark.stop()

## Dicas

- `count(*)` puro é instantâneo (estatísticas do Delta); filtros em colunas comuns leem os arquivos (segundos).
- O DuckDB lê **só as colunas pedidas** (Parquet é colunar): `SELECT *` em 73 milhões de linhas é o que deixa lento.
- Para guardar um resultado: `con.sql(...).df().to_csv(...)` em `/mnt/hdd1tb/lambda-dump/exports/` (nunca no repo).
- Nada aqui escreve no lake: escrita é papel do pipeline (um escritor por tabela).